# 1. Data Cleaning and Preparation:
##### 1. Removing duplicates and irrelevant columns.
##### 2. Handling missing values.
##### 3. Converting data types for columns such as dates and numeric values.
##### 4. Data validation: searching for illogical values

# Data cleaning and preparation of Calls

In [1]:
import numpy as np
import pandas as pd
import pickle
from geopandas.tools import geocode
import re

In [2]:
# ID columns have been converted to object to preserve values without changes.

calls = pd.read_excel('Calls (Done).xlsx', dtype={'Id': str, 'CONTACTID': str})

## Removing duplicates and irrelevant columns. Getting familiar with the data

In [3]:
calls.head()

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN


In [4]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Id                          95874 non-null  object 
 1   Call Start Time             95874 non-null  object 
 2   Call Owner Name             95874 non-null  object 
 3   CONTACTID                   91941 non-null  object 
 4   Call Type                   95874 non-null  object 
 5   Call Duration (in seconds)  95791 non-null  float64
 6   Call Status                 95874 non-null  object 
 7   Dialled Number              0 non-null      float64
 8   Outgoing Call Status        86875 non-null  object 
 9   Scheduled in CRM            86875 non-null  float64
 10  Tag                         0 non-null      float64
dtypes: float64(4), object(7)
memory usage: 8.0+ MB


In [5]:
# Bringing column names to a unified format for ease of analysis
calls.rename(columns={'Id': 'calls_id',
                      'Call Start Time': 'call_start_time',
                      'Call Owner Name': 'calls_manager',
                      'CONTACTID': 'contact_id',
                      'Call Type': 'call_type',
                      'Call Duration (in seconds)': 'call_duration_sec',
                      'Call Status': 'call_status',
                      'Outgoing Call Status': 'outgoing_call_status',
                      'Scheduled in CRM': 'scheduled_in_CRM',
                      'Dialled Number': 'dialled_number',
                      'Tag': 'tag'}, inplace=True)

In [6]:
# Checking for duplicates
calls.duplicated().sum()

np.int64(0)

In [7]:
calls[calls.notnull()].nunique()

calls_id                95874
call_start_time         68445
calls_manager              33
contact_id              15214
call_type                   3
call_duration_sec        2619
call_status                11
dialled_number              0
outgoing_call_status        4
scheduled_in_CRM            2
tag                         0
dtype: int64

In [8]:
# Checking for NaN values
calls.isna().sum()

calls_id                    0
call_start_time             0
calls_manager               0
contact_id               3933
call_type                   0
call_duration_sec          83
call_status                 0
dialled_number          95874
outgoing_call_status     8999
scheduled_in_CRM         8999
tag                     95874
dtype: int64

In [9]:
# Percentage of missing data by column

round(100 * calls.isnull().sum() / len(calls), 2)

calls_id                  0.00
call_start_time           0.00
calls_manager             0.00
contact_id                4.10
call_type                 0.00
call_duration_sec         0.09
call_status               0.00
dialled_number          100.00
outgoing_call_status      9.39
scheduled_in_CRM          9.39
tag                     100.00
dtype: float64

In [10]:
# Removing these fields dialled_number & tag since they are 100% missing

calls.drop(columns=['dialled_number','tag'], inplace=True)

## Handling missing values.

### сontact_id 

In [11]:
# Filling missing values in contact_id with a special flag '-1' and using it as an indicator that data is missing

calls['contact_id'] = calls['contact_id'].fillna('-1')

In [12]:
calls[calls['contact_id'] == '-1']

,calls_id,call_start_time,calls_manager,contact_id,call_type,call_duration_sec,call_status,outgoing_call_status,scheduled_in_CRM
0,5805028000000805001,30.06.2023 08:43,John Doe,-1,Inbound,171.0,Received,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,-1,Outbound,28.0,Attended Dialled,Completed,0.0
2,5805028000000764027,30.06.2023 08:59,John Doe,-1,Outbound,24.0,Attended Dialled,Completed,0.0
7,5805028000000879006,03.07.2023 13:06,Jane Smith,-1,Outbound,0.0,Unattended Dialled,Completed,0.0
8,5805028000000870005,03.07.2023 13:08,Jane Smith,-1,Outbound,40.0,Attended Dialled,Completed,0.0
...,...,...,...,...,...,...,...,...,...
95852,5805028000056849471,21.06.2024 15:06,Ulysses Adams,-1,Outbound,7.0,Attended Dialled,Completed,0.0
95853,5805028000056859477,21.06.2024 15:07,Ulysses Adams,-1,Inbound,407.0,Received,NaN,NaN
95861,5805028000056876318,21.06.2024 15:20,Eva Kent,-1,Outbound,5.0,Attended Dialled,Completed,0.0
95868,5805028000056912329,21.06.2024 15:30,Victor Barnes,-1,Outbound,NaN,Scheduled,Scheduled,1.0


### call_duration_sec

In [13]:
# Checking the dependence of missing values on call status
calls_duration_with_nan = calls[calls['call_duration_sec'].isnull()]
print(calls_duration_with_nan['call_status'].value_counts())

call_status
Overdue      60
Cancelled    20
Scheduled     3
Name: count, dtype: int64


In [14]:
# All missing values = unsuccessful calls -> fill with 0 (Scheduled can be ignored, as there are only 3 of them and they do not affect the overall picture)
calls['call_duration_sec'] = calls['call_duration_sec'].fillna(0)

In [15]:
round(100 * calls.isnull().sum() / len(calls), 2)

calls_id                0.00
call_start_time         0.00
calls_manager           0.00
contact_id              0.00
call_type               0.00
call_duration_sec       0.00
call_status             0.00
outgoing_call_status    9.39
scheduled_in_CRM        9.39
dtype: float64

### outgoing_call_status

In [16]:
calls['outgoing_call_status'].unique()

array([nan, 'Completed', 'Cancelled', 'Overdue', 'Scheduled'],
      dtype=object)

In [17]:
# Checking if the missing values are outgoing calls
calls[calls['outgoing_call_status'].isna()]['call_type'].unique()

array(['Inbound', 'Missed'], dtype=object)

The NaN values are logical because these calls were not outgoing.

### scheduled_in_CRM

In [18]:
# Checking the dependence of missing values on call status
nan_calls = calls[calls['scheduled_in_CRM'].isnull()]
print(nan_calls['call_status'].unique())
print(nan_calls['outgoing_call_status'].unique())

['Received' 'Missed']
[nan]


In [19]:
# Statuses ['Received', 'Missed'] in the call_status field and missing values in outgoing_call_status mean that the calls were not scheduled (no 'scheduled' status)
# Fill missing values in scheduled_in_CRM with 0, as these calls were not scheduled
calls['scheduled_in_CRM'] = calls['scheduled_in_CRM'].fillna(0)

In [20]:
round(100 * calls.isnull().sum() / len(calls), 2)

calls_id                0.00
call_start_time         0.00
calls_manager           0.00
contact_id              0.00
call_type               0.00
call_duration_sec       0.00
call_status             0.00
outgoing_call_status    9.39
scheduled_in_CRM        0.00
dtype: float64

## Converting to correct data types

In [21]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   calls_id              95874 non-null  object 
 1   call_start_time       95874 non-null  object 
 2   calls_manager         95874 non-null  object 
 3   contact_id            95874 non-null  object 
 4   call_type             95874 non-null  object 
 5   call_duration_sec     95874 non-null  float64
 6   call_status           95874 non-null  object 
 7   outgoing_call_status  86875 non-null  object 
 8   scheduled_in_CRM      95874 non-null  float64
dtypes: float64(2), object(7)
memory usage: 6.6+ MB


In [22]:
calls[calls.notnull()].nunique()

calls_id                95874
call_start_time         68445
calls_manager              33
contact_id              15215
call_type                   3
call_duration_sec        2619
call_status                11
outgoing_call_status        4
scheduled_in_CRM            2
dtype: int64

In [23]:
#Dates
calls['call_start_time'] = pd.to_datetime(calls['call_start_time'], errors='coerce', dayfirst=True)

#Numbers
calls['call_duration_sec'] = calls['call_duration_sec'].astype(int)


# Categories
for col in ['calls_manager', 'call_type', 'call_status', 'outgoing_call_status']:
    calls[col] = calls[col].astype('category')

#Boolean
calls['scheduled_in_CRM'] = calls['scheduled_in_CRM'].astype(bool)

In [24]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   calls_id              95874 non-null  object        
 1   call_start_time       95874 non-null  datetime64[ns]
 2   calls_manager         95874 non-null  category      
 3   contact_id            95874 non-null  object        
 4   call_type             95874 non-null  category      
 5   call_duration_sec     95874 non-null  int64         
 6   call_status           95874 non-null  category      
 7   outgoing_call_status  86875 non-null  category      
 8   scheduled_in_CRM      95874 non-null  bool          
dtypes: bool(1), category(4), datetime64[ns](1), int64(1), object(2)
memory usage: 3.4+ MB


## Data validation: searching for illogical values

In [25]:
# Checking for spaces in string fields that may cause errors during analysis

for col in calls.columns:
    if calls[col].dtype == 'object':  # only for string columns
        has_spaces = calls[col].str.contains('^\s|\s$', na=False).any()
        if has_spaces:
            print(f"'{col}' contains leading/trailing spaces")
            # cleaning immediately
            calls[col] = calls[col].str.strip()

In [26]:
calls['call_start_time'].describe()

count                            95874
mean     2024-02-05 12:34:44.369484800
min                2023-06-30 08:43:00
25%                2023-11-24 11:07:30
50%                2024-02-19 12:19:30
75%                2024-04-22 19:10:00
max                2024-06-21 15:31:00
Name: call_start_time, dtype: object

In [27]:
calls['call_status'].unique()

['Received', 'Attended Dialled', 'Unattended Dialled', 'Missed', 'Cancelled', ..., 'Overdue', 'Scheduled Unattended Delay', 'Scheduled Attended', 'Scheduled Attended Delay', 'Scheduled']
Length: 11
Categories (11, object): ['Attended Dialled', 'Cancelled', 'Missed', 'Overdue', ..., 'Scheduled Attended Delay', 'Scheduled Unattended', 'Scheduled Unattended Delay', 'Unattended Dialled']

In [28]:
# Checking for invalid records in call_duration_sec: duration > 0 with a status implying an unsuccessful call
calls[(calls['call_duration_sec'] > 0) & 
      (calls['call_status'].isin([
    'Unattended Dialled',      
    'Missed',                   
    'Cancelled',                
    'Overdue',                   
    'Scheduled Unattended',
    'Scheduled Unattended Delay'
]))]

,calls_id,call_start_time,calls_manager,contact_id,call_type,call_duration_sec,call_status,outgoing_call_status,scheduled_in_CRM


In [29]:
# Checking for values less than 0
calls[calls['call_duration_sec'] < 0]

,calls_id,call_start_time,calls_manager,contact_id,call_type,call_duration_sec,call_status,outgoing_call_status,scheduled_in_CRM


In [30]:

print(calls['calls_manager'].value_counts())

calls_manager
Yara Edwards       9059
Julia Nelson       7446
Ian Miller         7215
Charlie Davis      7213
Diana Evans        6857
Ulysses Adams      6085
Amy Green          5982
Nina Scott         5581
Victor Barnes      5439
Kevin Parker       5406
Paula Underwood    4580
Quincy Vincent     4384
Jane Smith         3753
Cara Iverson       3300
John Doe           2986
Ben Hall           2947
Alice Johnson      1251
Mason Roberts      1166
Derek James         948
George King         850
Zachary Foster      523
Eva Kent            498
Fiona Jackson       470
Sam Young           457
Rachel White        441
Xander Dean         304
Ethan Harris        280
Hannah Lee          175
Wendy Clark         162
Bob Brown            99
Oliver Taylor        10
Tina Zhang            5
Laura Quinn           2
Name: count, dtype: int64


In [31]:

print(calls['call_type'].value_counts())

call_type
Outbound    86875
Missed       5921
Inbound      3078
Name: count, dtype: int64


In [32]:
print(calls['call_status'].value_counts())

call_status
Attended Dialled              70703
Unattended Dialled            16030
Missed                         5922
Received                       3077
Overdue                          60
Scheduled Attended Delay         22
Cancelled                        20
Scheduled Unattended Delay       17
Scheduled Attended               14
Scheduled Unattended              6
Scheduled                         3
Name: count, dtype: int64


In [33]:
print(calls['outgoing_call_status'].value_counts())

outgoing_call_status
Completed    86792
Overdue         60
Cancelled       20
Scheduled        3
Name: count, dtype: int64


In [34]:
# Checking for data error: incoming calls cannot have an outgoing call status
calls[(calls['call_type'] == 'inbound')&(calls['outgoing_call_status'].notna())]

,calls_id,call_start_time,calls_manager,contact_id,call_type,call_duration_sec,call_status,outgoing_call_status,scheduled_in_CRM


# Data cleaning and preparation of Contacts

In [35]:
# Converting ID column to object to preserve values without changes.
contacts = pd.read_excel('Contacts (Done).xlsx', dtype={'Id': str})

## Removing duplicates and irrelevant columns. Getting familiar with the data

In [36]:
contacts

,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14
...,...,...,...,...
18543,5805028000056889209,Ulysses Adams,21.06.2024 12:11,21.06.2024 14:11
18544,5805028000056889351,Eva Kent,21.06.2024 13:32,21.06.2024 15:32
18545,5805028000056892018,Eva Kent,21.06.2024 10:21,21.06.2024 12:21
18546,5805028000056892055,Yara Edwards,21.06.2024 10:22,21.06.2024 12:23


In [37]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  object
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: object(4)
memory usage: 579.8+ KB


In [38]:
# Bringing column names to a unified format for ease of analysis

contacts.rename(columns={'Id': 'contact_id',
                      'Contact Owner Name': 'contact_manager',
                      'Created Time': 'created_time',
                      'Modified Time': 'modified_time',
                      }, inplace=True)

In [39]:
# Checking for duplicates

contacts.duplicated().sum()

np.int64(0)

In [40]:
contacts[contacts.notnull()].nunique()

contact_id         18548
contact_manager       28
created_time       17921
modified_time      16580
dtype: int64

## Handling missing values.

In [41]:
# Checking for NaN values

contacts.isna().sum()

contact_id         0
contact_manager    0
created_time       0
modified_time      0
dtype: int64

## Data validation: searching for illogical values

In [42]:
# Removing spaces

contacts['contact_manager'] = contacts['contact_manager'].str.strip()

In [43]:
# Checking for missing values
contacts.isnull().sum()

contact_id         0
contact_manager    1
created_time       0
modified_time      0
dtype: int64

In [44]:
contacts[contacts['contact_manager'].isnull()]

,contact_id,contact_manager,created_time,modified_time
2197,5805028000008772190,NaN,24.09.2023 09:01,13.10.2023 16:44


In [45]:
contacts = contacts.dropna(subset=['contact_manager'])


In [46]:
contacts.isnull().sum()

contact_id         0
contact_manager    0
created_time       0
modified_time      0
dtype: int64

In [47]:
contacts['contact_manager'].value_counts()

contact_manager
Charlie Davis      2018
Ulysses Adams      1816
Julia Nelson       1769
Paula Underwood    1487
Quincy Vincent     1416
Nina Scott         1150
Ben Hall           1038
Victor Barnes       967
Cara Iverson        880
Rachel White        782
Jane Smith          754
Bob Brown           685
Ian Miller          684
Diana Evans         678
Yara Edwards        655
Amy Green           621
Eva Kent            365
Kevin Parker        325
Mason Roberts       217
George King         144
Sam Young            37
Alice Johnson        27
Oliver Taylor        19
Zachary Foster        8
Wendy Clark           2
Tina Zhang            2
Derek James           1
Name: count, dtype: int64

In [48]:
# Checking for date errors: modified_time cannot be less than created_time

contacts[pd.to_datetime(contacts['modified_time']) < pd.to_datetime(contacts['created_time'])]

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2560960517.py:3: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  contacts[pd.to_datetime(contacts['modified_time']) < pd.to_datetime(contacts['created_time'])]
/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2560960517.py:3: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  contacts[pd.to_datetime(contacts['modified_time']) < pd.to_datetime(contacts['created_time'])]


,contact_id,contact_manager,created_time,modified_time


## Преобразуем типы данных в соответствии с их содержимым

In [49]:
#Dates
for col in ['created_time', 'modified_time']:
    contacts[col] = pd.to_datetime(contacts[col], errors='coerce', dayfirst=True)

# Category
contacts['contact_manager'] = contacts['contact_manager'].astype('category')

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/3539535523.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  contacts[col] = pd.to_datetime(contacts[col], errors='coerce', dayfirst=True)
/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/3539535523.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  contacts[col] = pd.to_datetime(contacts[col], errors='coerce', dayfirst=True)
/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/3539535523.py:6: SettingWithCopyWar

In [50]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18547 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   contact_id       18547 non-null  object        
 1   contact_manager  18547 non-null  category      
 2   created_time     18547 non-null  datetime64[ns]
 3   modified_time    18547 non-null  datetime64[ns]
dtypes: category(1), datetime64[ns](2), object(1)
memory usage: 599.0+ KB


# Data cleaning and preparation of Spend

In [51]:

spend = pd.read_excel('Spend (Done).xlsx')

## Removing duplicates and irrelevant columns.

In [52]:
spend

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20774,2024-06-21,Facebook Ads,17.03.24wide_AT,7,0.07,0,wide,bloggersvideo16com_at
20775,2024-06-21,Tiktok Ads,12.07.2023wide_DE,61,0.16,0,wide,bloggersvideo14com
20776,2024-06-21,Partnership,NaN,0,0.00,0,NaN,NaN
20777,2024-06-21,Tiktok Ads,NaN,0,0.00,0,NaN,NaN


In [53]:
spend.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20779 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         20779 non-null  datetime64[ns]
 1   Source       20779 non-null  object        
 2   Campaign     14785 non-null  object        
 3   Impressions  20779 non-null  int64         
 4   Spend        20779 non-null  float64       
 5   Clicks       20779 non-null  int64         
 6   AdGroup      13951 non-null  object        
 7   Ad           13951 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.3+ MB


In [54]:
spend.rename(columns={'Date': 'spend_date',
                      'Source': 'spend_source',
                      'Campaign': 'spend_campaign',
                      'Impressions': 'impressions',
                      'Spend': 'spend',
                      'Clicks': 'clicks',
                      'AdGroup': 'ad_group',
                      'Ad': 'ad'}, inplace=True)

In [55]:
spend.duplicated().sum()

np.int64(917)

In [56]:
# removing 917 duplicates
spend = spend.drop_duplicates()

## Handling missing values.

In [57]:
spend.isnull().sum()

spend_date           0
spend_source         0
spend_campaign    5077
impressions          0
spend                0
clicks               0
ad_group          5911
ad                5911
dtype: int64

In [58]:
round(100 * spend.isnull().sum() / len(spend), 2)

spend_date         0.00
spend_source       0.00
spend_campaign    25.56
impressions        0.00
spend              0.00
clicks             0.00
ad_group          29.76
ad                29.76
dtype: float64

### spend_campaign

In [59]:
spend['spend_campaign'].unique()

array(['gen_analyst_DE', 'performancemax_eng_DE', nan, '03.07.23women',
       '02.07.23wide_DE', '12.07.2023wide_DE', '05.07.23interests_DE',
       '04.07.23recentlymoved_DE', '07.07.23LAL_DE',
       '10.07.23wide_com_DE', '15.07.23b_DE', 'youtube_shorts_DE',
       '24.07.2023wide_DE', 'comp_search_DE', 'brand_search_eng_DE',
       '02.08.23interests_DE', 'web2408_DE', '05.09.2023wide_DE',
       '12.09.23interests_Uxui_DE', 'discovery_DE',
       '24.09.23retargeting_DE', '18.10.23wide_gos_DE',
       '14.11.23wide_webinar_DE', '15.11.23wide_webinar_DE', 'blog2_DE',
       '30.11.23wide_DE', '07.12.23test_DE', '01.02.24wide_webinar_DE',
       'bbo_DE', '09.02.24berlin_dd_DE', '17.03.24wide_AT',
       '15.03.24recentlymoved_AT', '15.03.2024wide_AT',
       'youtube_shortsin_AT', 'discovery_wide1_AT',
       'performancemax_wide_AT', '20.03.24_widde_PL', '20.03.2024wide_PL',
       '20.03.24interests_WebDev_AT', '20.03.24interests_WebDev_PL',
       'shorts_PL', '1performancemax_

In [60]:
# Filling missing values in spend_campaign with a special flag 'Unknown' and using it as an indicator that data is missing
spend['spend_campaign'] = spend['spend_campaign'].fillna('Unknown')

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2107226375.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spend['spend_campaign'] = spend['spend_campaign'].fillna('Unknown')


### ad_group

In [61]:
spend['ad_group'].unique()


array([nan, 'women', 'wide', 'interest_programming', 'recentlymoved',
       'interest_dataanalytics', 'interest_work',
       'interest_programming – Copy', 'interest_dataanalytics – Copy',
       'LAL1', 'b', 'Com_july_1', 'interest_all', 'Com_august',
       'interest_work_WebDev', 'interest_programming_WebDev',
       'promoposts_b', 'retargeting', 'wide_webdesigner',
       'wide_python-developer', 'wide_qa-engineer',
       'interest_python-developer', 'berlin_wide', 'Com_march',
       'accountant_wide'], dtype=object)

In [62]:
# Insufficient information to fill missing values in ad_group, so filling them with a special flag 'Unknown' and using it as an indicator that data is missing

spend['ad_group'] = spend['ad_group'].fillna('Unknown')

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/1698858557.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spend['ad_group'] = spend['ad_group'].fillna('Unknown')


### ad

In [63]:
spend['ad'].unique()

array([nan, 'b3', 'b1', 'b4', 'b2', 'v2', 'v1', 'b4com', 'b3com', 'b2com',
       'b1com', 'v6com', 'v5', 'v4com', 'v3com', 'v5com', 'ad4', 'ad1',
       'ad2', 'ad3', 'v8com', 'v7com', 'ad6', 'ad5', 'bloggersvideo1com',
       'v9com', 'ad9', 'ad8', 'ad_blogger_1', 'ad_blogger_2', 'ad7',
       'web_b3', 'web_b5', 'web_b1', 'web_b4', 'web_b2', 'ad_blogger_3',
       'v10com', 'bloggersvideo2com', 'b5', 'b6', 'b8', 'b7', 'v3', 'v10',
       'v12', 'v11com', 'v11', 'ad_gov_1', 'ad_da_1', 'b3comwebdev',
       'bloggersvideo2comwebdev', 'v11comwebdev', 'b1comwebdev',
       'b2comwebdev', 'bloggersvideo4com', 'bloggersvideo3com',
       'bloggersvideo5', 'promo2', 'promo1', 'ad_blogger_4',
       'bloggersvideo4', 'b10', 'b11', 'b12', 'ad_blogger_6', 'promo3',
       'b15blackfriday', 'b14blackfriday', 'b13blackfriday', 'b7webinar',
       'b6webinar', 'b4webinar', 'b5webinar', 'bloggersvideo6blackfriday',
       'bloggersvideo6webinar', 'bloggersvideo7blackfriday',
       'bloggersvideo

In [64]:
# Insufficient information to fill missing values in ad, so filling them with a special flag 'Unknown' and using it as an indicator that data is missing

spend['ad'] = spend['ad'].fillna('Unknown')

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2010899659.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spend['ad'] = spend['ad'].fillna('Unknown')


## Converting data types for columns such as dates and numeric values.

In [65]:
spend.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19862 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   spend_date      19862 non-null  datetime64[ns]
 1   spend_source    19862 non-null  object        
 2   spend_campaign  19862 non-null  object        
 3   impressions     19862 non-null  int64         
 4   spend           19862 non-null  float64       
 5   clicks          19862 non-null  int64         
 6   ad_group        19862 non-null  object        
 7   ad              19862 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.4+ MB


In [66]:
spend.head()

,spend_date,spend_source,spend_campaign,impressions,spend,clicks,ad_group,ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,Unknown,Unknown
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,Unknown,Unknown
2,2023-07-03,Facebook Ads,Unknown,0,0.00,0,Unknown,Unknown
3,2023-07-03,Google Ads,Unknown,0,0.00,0,Unknown,Unknown
4,2023-07-03,CRM,Unknown,0,0.00,0,Unknown,Unknown


In [67]:
spend.nunique()

spend_date         355
spend_source        14
spend_campaign      52
impressions       4003
spend             2859
clicks             552
ad_group            25
ad                 177
dtype: int64

In [68]:
# Category

for col in ['spend_source', 'spend_campaign', 'ad_group']:
    spend[col] = spend[col].astype('category')

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2885529565.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spend[col] = spend[col].astype('category')
/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2885529565.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spend[col] = spend[col].astype('category')
/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2885529565.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFra

In [69]:
spend.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19862 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   spend_date      19862 non-null  datetime64[ns]
 1   spend_source    19862 non-null  category      
 2   spend_campaign  19862 non-null  category      
 3   impressions     19862 non-null  int64         
 4   spend           19862 non-null  float64       
 5   clicks          19862 non-null  int64         
 6   ad_group        19862 non-null  category      
 7   ad              19862 non-null  object        
dtypes: category(3), datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 993.1+ KB


## Data validation: searching for illogical values

In [70]:
# Checking for negative values in numeric fields that may be data errors

for col in ['spend', 'clicks', 'impressions']:
    print(spend[spend[col] < 0])

Empty DataFrame
Columns: [spend_date, spend_source, spend_campaign, impressions, spend, clicks, ad_group, ad]
Index: []
Empty DataFrame
Columns: [spend_date, spend_source, spend_campaign, impressions, spend, clicks, ad_group, ad]
Index: []
Empty DataFrame
Columns: [spend_date, spend_source, spend_campaign, impressions, spend, clicks, ad_group, ad]
Index: []


In [71]:
spend['ad_group'].value_counts()

ad_group
Unknown                          5911
wide                             5451
recentlymoved                    1442
women                            1274
LAL1                             1220
Com_august                       1073
interest_work_WebDev              733
interest_programming_WebDev       636
b                                 566
retargeting                       504
Com_march                         206
interest_work                     181
Com_july_1                        150
interest_programming              121
promoposts_b                       71
wide_python-developer              56
wide_qa-engineer                   50
berlin_wide                        48
wide_webdesigner                   42
interest_all                       36
interest_python-developer          28
interest_dataanalytics             27
accountant_wide                    21
interest_programming – Copy         8
interest_dataanalytics – Copy       7
Name: count, dtype: int64

In [72]:
# Checking in google sheets showed that the rows with 'interest_dataanalytics – Copy' and 'interest_programming – Copy' 
# are not duplicates of 'interest_dataanalytics' and 'interest_programming' respectively, so we remove Copy from the names

spend['ad_group'] = spend['ad_group'].replace({'interest_dataanalytics – Copy': 'interest_dataanalytics',
                                               'interest_programming – Copy': 'interest_programming '})

/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2850100032.py:4: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  spend['ad_group'] = spend['ad_group'].replace({'interest_dataanalytics – Copy': 'interest_dataanalytics',
/var/folders/b2/7j01xs_d4yvgcqyrhxv8x00w0000gn/T/ipykernel_34960/2850100032.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spend['ad_group'] = spend['ad_group'].replace({'interest_dataanalytics – Copy': 'interest_dataanalytics',


In [73]:
spend['ad_group'].value_counts()

ad_group
Unknown                        5911
wide                           5451
recentlymoved                  1442
women                          1274
LAL1                           1220
Com_august                     1073
interest_work_WebDev            733
interest_programming_WebDev     636
b                               566
retargeting                     504
Com_march                       206
interest_work                   181
Com_july_1                      150
interest_programming            121
promoposts_b                     71
wide_python-developer            56
wide_qa-engineer                 50
berlin_wide                      48
wide_webdesigner                 42
interest_all                     36
interest_dataanalytics           34
interest_python-developer        28
accountant_wide                  21
interest_programming              8
Name: count, dtype: int64

# Data cleaning and preparation of Deals

In [74]:
# ID columns have been converted to object to preserve values without changes.

deals = pd.read_excel('Deals (Done).xlsx', dtype={'Id': str, 'Contact Name': str, 'SLA': str})

## Removing duplicates and irrelevant columns.

In [75]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21595 entries, 0 to 21594
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   21593 non-null  object 
 1   Deal Owner Name      21564 non-null  object 
 2   Closing Date         14645 non-null  object 
 3   Quality              19340 non-null  object 
 4   Stage                21593 non-null  object 
 5   Lost Reason          16124 non-null  object 
 6   Page                 21593 non-null  object 
 7   Campaign             16067 non-null  object 
 8   SLA                  15533 non-null  object 
 9   Content              14147 non-null  object 
 10  Term                 12454 non-null  object 
 11  Source               21593 non-null  object 
 12  Payment Type         496 non-null    object 
 13  Product              3592 non-null   object 
 14  Education Type       3300 non-null   object 
 15  Created Time         21593 non-null 

In [76]:
deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN


In [77]:
# Bringing column names to a unified format for ease of analysis
deals.rename(columns={'Id': 'deals_id',
                      'Deal Owner Name': 'deal_manager',
                      'Closing Date': 'closing_date',
                      'Quality': 'quality',
                      'Stage': 'stage',
                      'Lost Reason': 'lost_reason',
                      'Page': 'page',
                      'Campaign': 'deal_campaign',
                      'SLA': 'sla',
                      'Content': 'content',
                      'Term': 'term',
                      'Source': 'source',
                      'Payment Type': 'payment_type',
                      'Product': 'product',
                      'Education Type': 'education_type',
                      'Created Time': 'created_time',
                      'Course duration': 'course_duration',
                      'Months of study': 'months_of_study',
                      'Initial Amount Paid': 'initial_amount_paid',
                      'Offer Total Amount': 'offer_total_amount',
                      'Contact Name': 'contact_name',
                      'City': 'city',
                      'Level of Deutsch': 'level_of_deutsch'}, inplace=True)

In [78]:
# Checking for duplicates

deals.duplicated().sum()

np.int64(0)

## Handling missing values.

In [79]:
# Checking for NaN values
deals.isna().sum()

deals_id                   2
deal_manager              31
closing_date            6950
quality                 2255
stage                      2
lost_reason             5471
page                       2
deal_campaign           5528
sla                     6062
content                 7448
term                    9141
source                     2
payment_type           21099
product                18003
education_type         18295
created_time               2
course_duration        18008
months_of_study        20755
initial_amount_paid    17430
offer_total_amount     17410
contact_name              63
city                   19084
level_of_deutsch       20344
dtype: int64

In [80]:
round(100 * deals.isnull().sum() / len(deals), 2)

deals_id                0.01
deal_manager            0.14
closing_date           32.18
quality                10.44
stage                   0.01
lost_reason            25.33
page                    0.01
deal_campaign          25.60
sla                    28.07
content                34.49
term                   42.33
source                  0.01
payment_type           97.70
product                83.37
education_type         84.72
created_time            0.01
course_duration        83.39
months_of_study        96.11
initial_amount_paid    80.71
offer_total_amount     80.62
contact_name            0.29
city                   88.37
level_of_deutsch       94.21
dtype: float64

### deals_id

In [81]:
# Removing rows with missing deals_id, as this is the key deal identifier and its absence makes the record useless for analysis

deals = deals.dropna(subset=['deals_id'])

### deal_manager

In [82]:
deals[deals['deal_manager'].isna()]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
14007,5805028000022036007,NaN,01.07.2023,E - Non Qualified,Payment Done,NaN,/,NaN,NaN,NaN,...,NaN,NaN,19.12.2023 10:37,NaN,NaN,0,0,NaN,NaN,NaN
18834,5805028000009465153,NaN,NaN,C - Low,Qualificated,NaN,eng/digital-marketing,12.07.2023wide_DE,00:28:20,bloggersvideo3com,...,NaN,NaN,29.09.2023 18:09,NaN,NaN,NaN,NaN,5805028000009482152,NaN,NaN
18954,5805028000009133444,NaN,NaN,C - Low,Qualificated,NaN,/eng,24.09.23retargeting_DE,02:15:53,bloggersvideo5,...,NaN,NaN,27.09.2023 13:23,NaN,NaN,NaN,NaN,5805028000009150146,NaN,NaN
19054,5805028000008812739,NaN,NaN,C - Low,Qualificated,NaN,eng/digital-marketing,04.07.23recentlymoved_DE,00:03:30,bloggersvideo4com,...,NaN,NaN,25.09.2023 15:11,NaN,NaN,NaN,NaN,5805028000008758890,NaN,NaN
19150,5805028000008666416,NaN,12.10.2023,C - Low,Lost,Changed Decision,/direct,NaN,00:11:32,NaN,...,NaN,NaN,23.09.2023 16:34,NaN,NaN,NaN,NaN,5805028000008647384,NaN,NaN
19248,5805028000008355196,NaN,NaN,B - Medium,Qualificated,NaN,/eng,mu_DE,00:09:00,NaN,...,NaN,NaN,21.09.2023 11:32,NaN,NaN,NaN,NaN,5805028000008364208,NaN,NaN
19254,5805028000008367218,NaN,NaN,C - Low,Qualificated,NaN,/eng/test,mu_DE,11:47:45,NaN,...,NaN,NaN,21.09.2023 08:50,NaN,NaN,NaN,NaN,5805028000008367188,NaN,NaN
19337,5805028000008036373,NaN,NaN,B - Medium,Qualificated,NaN,eng/digital-marketing,15.07.23b_DE,02:28:27,b6,...,NaN,NaN,19.09.2023 14:43,NaN,NaN,NaN,NaN,5805028000008017270,NaN,NaN
19382,5805028000007956266,NaN,NaN,C - Low,Qualificated,NaN,eng/digital-marketing,05.09.2023wide_DE,15:18:30,v11,...,NaN,NaN,18.09.2023 04:02,NaN,NaN,NaN,NaN,5805028000007865975,NaN,NaN
19438,5805028000007863693,NaN,13.10.2023,C - Low,Lost,Not for myself,/eng/ux-ui,Dis_DE,00:09:01,151836595805_{region_name}_673801337002,...,NaN,NaN,16.09.2023 10:45,NaN,NaN,NaN,NaN,5805028000007854438,NaN,NaN


In [83]:
# Given that the number of missing values in the deal_manager column is a small fraction of the total data (0.14%),
# and in the columns important for analysis [course_duration, education_type, product, offer_total_amount, initial_amount_paid] data is missing,
# we remove rows with missing values in this column, as this is key information for analysis and its absence makes the record useless for analysis

deals = deals.dropna(subset=['deal_manager'])

In [84]:
round(100 * deals.isnull().sum() / len(deals), 2)

deals_id                0.00
deal_manager            0.00
closing_date           32.12
quality                10.45
stage                   0.00
lost_reason            25.26
page                    0.00
deal_campaign          25.62
sla                    28.04
content                34.51
term                   42.35
source                  0.00
payment_type           97.70
product                83.34
education_type         84.70
created_time            0.00
course_duration        83.37
months_of_study        96.10
initial_amount_paid    80.76
offer_total_amount     80.67
contact_name            0.28
city                   88.36
level_of_deutsch       94.20
dtype: float64

### quality

In [85]:
deals[deals['quality'].isna()]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
5,5805028000056828429,Paula Underwood,NaN,NaN,Need a consultation,NaN,/eng,youtube_shorts_DE,01:33:10,bloggersvideo2june,...,NaN,NaN,21.06.2024 13:02,NaN,NaN,NaN,NaN,5805028000056833279,NaN,NaN
6,5805028000056893379,Ulysses Adams,NaN,NaN,Need To Call,NaN,eng/digital-marketing,NaN,NaN,NaN,...,NaN,NaN,21.06.2024 12:52,NaN,NaN,NaN,NaN,5805028000056832215,NaN,NaN
7,5805028000056849262,Eva Kent,NaN,NaN,Need a consultation,NaN,/eng,brand_search_eng_DE,02:12:29,152789402780_{region_name}_695563281558,...,NaN,NaN,21.06.2024 12:44,NaN,NaN,NaN,NaN,5805028000056833242,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20371,5805028000005178088,John Doe,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,NaN,18.08.2023 20:46,NaN,NaN,NaN,NaN,5805028000000872003,NaN,NaN
20373,5805028000005176076,John Doe,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,Digital Marketing,Evening,18.08.2023 20:22,11.0,NaN,NaN,NaN,5805028000000872003,NaN,NaN
20375,5805028000005174051,John Doe,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,NaN,18.08.2023 19:54,NaN,NaN,NaN,NaN,5805028000000872003,NaN,NaN
20376,5805028000005168037,John Doe,NaN,NaN,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,NaN,...,NaN,NaN,18.08.2023 19:45,NaN,NaN,NaN,NaN,5805028000000872003,NaN,NaN


In [86]:
(deals['quality'].value_counts())

quality
E - Non Qualified    7633
D - Non Target       6248
C - Low              3442
B - Medium           1556
A - High              429
F                       3
Name: count, dtype: int64

In [87]:
deals.groupby('quality')['stage'].value_counts()

quality            stage                
A - High           Lost                      235
                   Payment Done              143
                   Call Delayed               24
                   Waiting For Payment        23
                   Test Sent                   3
                   Qualificated                1
B - Medium         Lost                     1009
                   Payment Done              327
                   Waiting For Payment       100
                   Call Delayed               89
                   Qualificated               22
                   Test Sent                   8
                   Need to Call - Sales        1
C - Low            Lost                     2554
                   Payment Done              355
                   Call Delayed              314
                   Waiting For Payment       138
                   Qualificated               68
                   Test Sent                   9
                   Need to C

In [88]:
deals.groupby('stage')['quality'].value_counts()

stage                  quality          
Call Delayed           D - Non Target       1804
                       C - Low               314
                       B - Medium             89
                       A - High               24
                       E - Non Qualified      17
Free Education         C - Low                 1
Lost                   E - Non Qualified    7607
                       D - Non Target       4328
                       C - Low              2554
                       B - Medium           1009
                       A - High              235
                       F                       3
Need a consultation    D - Non Target          1
Need to Call - Sales   C - Low                 3
                       D - Non Target          2
                       B - Medium              1
                       E - Non Qualified       1
Payment Done           C - Low               355
                       B - Medium            327
                       A - H

In [89]:
# Checking if there is a relationship between missing values in quality and stage/lost_reason to understand if these missing values can be filled based on these fields
quality_nan_deals = deals[deals['quality'].isna()]
print(quality_nan_deals['stage'].unique())
print(quality_nan_deals['lost_reason'].unique())

['New Lead' 'Need a consultation' 'Need To Call' 'Registered on Webinar'
 'Need to Call - Sales' 'Lost' 'Registered on Offline Day']
[nan 'Expensive' 'Duplicate']


In [90]:
pd.crosstab(
    deals['stage'].fillna('missing'),
    deals['quality'].fillna('missing')
)

quality,A - High,B - Medium,C - Low,D - Non Target,E - Non Qualified,F,missing
stage,,,,,,,
Call Delayed,24,89,314,1804,17,0,0
Free Education,0,0,1,0,0,0,0
Lost,235,1009,2554,4328,7607,3,1
Need To Call,0,0,0,0,0,0,31
Need a consultation,0,0,0,1,0,0,22
Need to Call - Sales,0,1,3,2,1,0,26
New Lead,0,0,0,0,0,0,6
Payment Done,143,327,355,31,1,0,0
Qualificated,1,22,68,14,1,0,0


In [91]:
# F almost always appears in Lost and is close in meaning to E - Non Qualified, given the number of records (3)
# we decide to replace F with E - Non Qualified to simplify analysis and improve data quality, as these records 
# do not provide additional information and may be the result of a data error.

deals['quality'] = deals['quality'].replace({'F': 'E - Non Qualified'})

In [92]:
# For missing values in quality, they almost always occur in Registered on Webinar and Registered on Offline Day, which indicates that
# these deals were most likely not qualified, since registration for a webinar or offline day is usually an initial stage, and we can assume that
# the manager has no understanding of the lead's further intentions, so for further analysis we replace empty values in quality with 'F - Unknown'

deals['quality'] = deals['quality'].fillna('F - Unknown')

### lost_reason

In [93]:
 deals['lost_reason'].value_counts(dropna=False)

lost_reason
NaN                                        5446
Doesn't Answer                             4134
Changed Decision                           2143
Duplicate                                  1771
Non target                                 1761
Stopped Answering                          1588
Invalid number                             1481
needs time to think                         655
Expensive                                   626
Conditions are not suitable                 531
Next stream                                 288
Inadequate                                  176
Gutstein refusal                            172
Considering a different direction in IT     148
Not for myself                              144
Does not speak English                      138
Didn't leave an application                 133
Thought for free                            110
Does not know how to use a computer          50
Went to Rivals                               47
The contract did not fit    

In [94]:
deals[deals['lost_reason'].isnull()]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
0,5805028000056864695,Ben Hall,NaN,F - Unknown,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,F - Unknown,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
5,5805028000056828429,Paula Underwood,NaN,F - Unknown,Need a consultation,NaN,/eng,youtube_shorts_DE,01:33:10,bloggersvideo2june,...,NaN,NaN,21.06.2024 13:02,NaN,NaN,NaN,NaN,5805028000056833279,NaN,NaN
6,5805028000056893379,Ulysses Adams,NaN,F - Unknown,Need To Call,NaN,eng/digital-marketing,NaN,NaN,NaN,...,NaN,NaN,21.06.2024 12:52,NaN,NaN,NaN,NaN,5805028000056832215,NaN,NaN
7,5805028000056849262,Eva Kent,NaN,F - Unknown,Need a consultation,NaN,/eng,brand_search_eng_DE,02:12:29,152789402780_{region_name}_695563281558,...,NaN,NaN,21.06.2024 12:44,NaN,NaN,NaN,NaN,5805028000056833242,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21399,5805028000001885195,Charlie Davis,NaN,C - Low,Call Delayed,NaN,eng/digital-marketing,07.07.23LAL_DE,00:12:22,b3,...,NaN,NaN,16.07.2023 09:53,NaN,NaN,0,0,5805028000001857349,NaN,NaN
21410,5805028000001885076,Jane Smith,31.08.2023,A - High,Payment Done,NaN,eng/digital-marketing,04.07.23recentlymoved_DE,00:13:31,b2,...,Digital Marketing,Morning,15.07.2023 13:27,11.0,11.0,450,4000,5805028000001880249,Ingolstadt,NaN
21555,5805028000001401001,Oliver Taylor,NaN,B - Medium,Payment Done,NaN,eng/digital-marketing,02.07.23wide_DE,02:22:36,b3,...,Digital Marketing,NaN,08.07.2023 08:56,11.0,8.0,1000,11500,5805028000001350049,NaN,NaN
21585,5805028000000935081,Julia Nelson,NaN,D - Non Target,Call Delayed,NaN,eng/digital-marketing,03.07.23women,"70 days, 23:12:53",b3,...,Digital Marketing,Morning,04.07.2023 11:46,11.0,NaN,1000,11500,5805028000000971007,NaN,NaN


In [95]:
print(deals['lost_reason'].unique())

[nan 'Non target' 'Invalid number' 'Duplicate' 'Inadequate' 'Expensive'
 'needs time to think' 'Not for myself'
 'Considering a different direction in IT' "Doesn't Answer"
 'Changed Decision' 'The contract did not fit' 'Stopped Answering'
 'Gutstein refusal' "Didn't leave an application"
 'Does not know how to use a computer' 'Conditions are not suitable'
 'Thought for free' 'Does not speak English' 'Went to Rivals'
 'Next stream' 'Refugee']


In [96]:
# Checking if there is a relationship between missing values in lost_reason and stage to understand if these missing values can be filled based on these fields

# deals_nan_lost_reason = deals[deals['lost_reason'].isnull()]
# print(deals_nan_lost_reason['stage'].unique())

pd.crosstab(
    deals['lost_reason'].fillna('missing'),
    deals['stage'].fillna('missing')
)

# The missing values in lost_reason look logical since most of them are associated with stages that 
# do not imply deal loss (Registered on Webinar, Registered on Offline Day, Trial Lesson), except for 47 missing values
# in lost_reason that are associated with the Lost stage, which may be the result of a data error, so we will fill these missing values with 'Unknown'.

stage,Call Delayed,Free Education,Lost,Need To Call,Need a consultation,Need to Call - Sales,New Lead,Payment Done,Qualificated,Registered on Offline Day,Registered on Webinar,Test Sent,Waiting For Payment
lost_reason,,,,,,,,,,,,,
Changed Decision,12,0,2119,0,0,0,0,7,2,0,0,0,3
Conditions are not suitable,3,0,524,0,0,0,0,4,0,0,0,0,0
Considering a different direction in IT,0,0,148,0,0,0,0,0,0,0,0,0,0
Didn't leave an application,2,0,131,0,0,0,0,0,0,0,0,0,0
Does not know how to use a computer,0,0,49,0,0,0,0,1,0,0,0,0,0
Does not speak English,0,0,138,0,0,0,0,0,0,0,0,0,0
Doesn't Answer,38,0,4073,0,0,1,0,15,0,0,0,2,5
Duplicate,5,0,1746,0,0,0,0,2,0,15,2,0,1
Expensive,5,0,614,0,0,0,0,6,0,0,0,0,1


In [97]:
# The missing values in lost_reason look logical since most of them are associated with stages that 
# do not imply deal loss (Registered on Webinar, Registered on Offline Day, Trial Lesson), except for 47 missing values
# in lost_reason that are associated with the Lost stage, which may be the result of a data error, so we will fill these missing values with 'Unknown'.

deals.loc[(deals['stage'].str.strip() == 'Lost') & (deals['lost_reason'].isnull()), 'lost_reason'] = 'Unknown'

In [98]:
pd.crosstab(
    deals['lost_reason'].fillna('missing'),
    deals['stage'].fillna('missing')
)

stage,Call Delayed,Free Education,Lost,Need To Call,Need a consultation,Need to Call - Sales,New Lead,Payment Done,Qualificated,Registered on Offline Day,Registered on Webinar,Test Sent,Waiting For Payment
lost_reason,,,,,,,,,,,,,
Changed Decision,12,0,2119,0,0,0,0,7,2,0,0,0,3
Conditions are not suitable,3,0,524,0,0,0,0,4,0,0,0,0,0
Considering a different direction in IT,0,0,148,0,0,0,0,0,0,0,0,0,0
Didn't leave an application,2,0,131,0,0,0,0,0,0,0,0,0,0
Does not know how to use a computer,0,0,49,0,0,0,0,1,0,0,0,0,0
Does not speak English,0,0,138,0,0,0,0,0,0,0,0,0,0
Doesn't Answer,38,0,4073,0,0,1,0,15,0,0,0,2,5
Duplicate,5,0,1746,0,0,0,0,2,0,15,2,0,1
Expensive,5,0,614,0,0,0,0,6,0,0,0,0,1


### deal_campaign

In [99]:
deals.isna().sum()

deals_id                   0
deal_manager               0
closing_date            6926
quality                    0
stage                      0
lost_reason             5399
page                       0
deal_campaign           5524
sla                     6046
content                 7441
term                    9133
source                     0
payment_type           21069
product                17972
education_type         18265
created_time               0
course_duration        17977
months_of_study        20724
initial_amount_paid    17415
offer_total_amount     17395
contact_name              60
city                   19053
level_of_deutsch       20313
dtype: int64

In [100]:
# the most frequent campaign for each combination 

mapping = {}

for (term, source), group in deals[deals['deal_campaign'].notnull()].groupby(['term', 'source']):
    # Find the most common campaign
    most_common = group['deal_campaign'].mode()
    if len(most_common) > 0:
        mapping[(term, source)] = most_common.iloc[0]
    else:
        mapping[(term, source)] = None

print("Mapping term-source -> campaign:")
for key, value in mapping.items():
    print(f"{key} -> {value}")

Mapping term-source -> campaign:
('01_02_2024', 'Telegram posts') -> jet_DE
('01_04_2024', 'Telegram posts') -> jet_DE
('01_05_2024', 'Bloggers') -> bloggerfrai_DE
('01_08_2023', 'Telegram posts') -> lviv_DE
('01_09_2023', 'Telegram posts') -> arina_DE
('01_10_2023', 'Telegram posts') -> jet_DE
('01_11_2023', 'Telegram posts') -> jet_DE
('02_04_2024', 'Telegram posts') -> Aussiedler_DE
('03_01_2024', 'Bloggers') -> bloggerfrai_DE
('03_05_2024', 'Telegram posts') -> uk12_AT
('03_06_2024', 'Telegram posts') -> Live_DE
('04_03_2024', 'SMM') -> talav_DE
('04_06_2024', 'Telegram posts') -> Markt_DE
('04_08_2023', 'Telegram posts') -> karta_DE
('04_09_2023', 'Bloggers') -> bbo_DE
('04_09_2023', 'Telegram posts') -> grad_DE
('04_10_2023', 'Telegram posts') -> nochtum_DE
('04_12_2023', 'Telegram posts') -> Bloggerel_DE
('05_03_2024', 'Telegram posts') -> uk_DE
('05_06_2024', 'SMM') -> bloggerdr_DE
('05_10_2023', 'Bloggers') -> BloggerShina_DE
('05_10_2023', 'Telegram posts') -> chat_DE
('06_02

In [101]:
# # for each term + source combination we find the most frequent campaign and fill missing values in deal_campaign based on this mapping, 
# # if there is no data for the combination, we leave the missing value since missing values for the remaining combinations look logical and may be the result of missing data rather than a data error

deals['deal_campaign'] = deals['deal_campaign'].fillna(
    deals.groupby(['term', 'source'])['deal_campaign'].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None))

In [102]:
# checking the result using an example of one record that had a missing value in deal_campaign and a term + source combination for which it was 'youtube_shorts_DE'

deals[deals['deals_id'] == '5805028000053387096']

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
1431,5805028000053387096,Cara Iverson,NaN,D - Non Target,Call Delayed,NaN,/eng/test,youtube_shorts_DE,01:29:04,bloggersvideo10,...,NaN,NaN,04.06.2024 15:03,NaN,NaN,NaN,NaN,5805028000053381152,NaN,NaN


In [103]:
# 359 missing values in deal_campaign have been filled based on the term + source combination, which looks logical, since for these combinations there is data and the missing values 
# could have been the result of a data error rather than a logical lack of information
# The remaining missing values in content look logical, since for these combinations there is no data and the missing values could be the result of missing data rather than a data error

deals.isna().sum()

deals_id                   0
deal_manager               0
closing_date            6926
quality                    0
stage                      0
lost_reason             5399
page                       0
deal_campaign           5165
sla                     6046
content                 7441
term                    9133
source                     0
payment_type           21069
product                17972
education_type         18265
created_time               0
course_duration        17977
months_of_study        20724
initial_amount_paid    17415
offer_total_amount     17395
contact_name              60
city                   19053
level_of_deutsch       20313
dtype: int64

### content

In [104]:
deals['content'].value_counts(dropna=False)

content
NaN                                        7441
_{region_name}_                            3255
bloggersvideo9com                          1523
bloggersvideo4com                           815
bloggersvideo5                              775
                                           ... 
bloggersvideo2comwebdev                       1
ad5                                           1
157624907008_{region_name}_668024583827       1
ad3                                           1
b1                                            1
Name: count, Length: 188, dtype: int64

In [105]:
# Filling missing values in content based on the term + source + deal_campaign combination,
# using the most frequent content for each combination if it exists, otherwise leaving the missing value, 
# since for the remaining combinations there is no data and the missing values could be the result of missing data rather than a data error

deals['content'] = deals['content'].fillna(
    deals.groupby(['term', 'source', 'deal_campaign'])['content']
         .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
)

In [106]:
# Only 2 records in content have been filled based on the term + source + deal_campaign combination.
# The remaining missing values in content look logical, since for these combinations there is no data and the missing values could be the result of missing data rather than a data error

deals.isna().sum()

deals_id                   0
deal_manager               0
closing_date            6926
quality                    0
stage                      0
lost_reason             5399
page                       0
deal_campaign           5165
sla                     6046
content                 7439
term                    9133
source                     0
payment_type           21069
product                17972
education_type         18265
created_time               0
course_duration        17977
months_of_study        20724
initial_amount_paid    17415
offer_total_amount     17395
contact_name              60
city                   19053
level_of_deutsch       20313
dtype: int64

### term

In [107]:
# Filling missing values in term based on the content + source + deal_campaign combination,
# using the most frequent term for each combination if it exists, otherwise leaving the missing value

deals['term'] = deals['term'].fillna(
    deals.groupby(['content', 'source', 'deal_campaign'])['term']
         .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
)

In [108]:
deals.isna().sum()

deals_id                   0
deal_manager               0
closing_date            6926
quality                    0
stage                      0
lost_reason             5399
page                       0
deal_campaign           5165
sla                     6046
content                 7439
term                    9133
source                     0
payment_type           21069
product                17972
education_type         18265
created_time               0
course_duration        17977
months_of_study        20724
initial_amount_paid    17415
offer_total_amount     17395
contact_name              60
city                   19053
level_of_deutsch       20313
dtype: int64

In [109]:
deals['term'].nunique()

220

### initial_amount_paid

In [110]:
deals['initial_amount_paid'].unique()

array([nan, 0, 1000, '€ 3.500,00', 500, 100, 4500, 300, 200, 2000, 11000,
       4000, 3000, 3500, 11500, 1200, 1500, 1, 5000, 600, 700, 350, 9,
       400, 450], dtype=object)

In [111]:
# Converting initial_amount_paid to numeric format by removing currency symbols and replacing commas with dots, then converting to Int64 to preserve missing values
# Replacing 'nan' with np.nan, since after converting to string, missing values became the string 'nan'

deals['initial_amount_paid'] = (
    deals['initial_amount_paid']
    .astype(str)
    .str.strip()
    .str.replace('€', '')
    .str.replace('.', '', regex=False)
    .str.replace(',', '.')
    .replace('nan', np.nan))

In [112]:
# Converting initial_amount_paid to numeric type, invalid values will be replaced with NaN, then converting to Int64 to preserve missing values

deals['initial_amount_paid'] = pd.to_numeric(deals['initial_amount_paid'], errors='coerce')
deals['initial_amount_paid'] = deals['initial_amount_paid'].astype('Int64')

In [113]:
# Checking the result

deals['initial_amount_paid'].unique()

<IntegerArray>
[ <NA>,     0,  1000,  3500,   500,   100,  4500,   300,   200,  2000, 11000,
  4000,  3000, 11500,  1200,  1500,     1,  5000,   600,   700,   350,     9,
   400,   450]
Length: 24, dtype: Int64

In [114]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21564 entries, 0 to 21592
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   deals_id             21564 non-null  object 
 1   deal_manager         21564 non-null  object 
 2   closing_date         14638 non-null  object 
 3   quality              21564 non-null  object 
 4   stage                21564 non-null  object 
 5   lost_reason          16165 non-null  object 
 6   page                 21564 non-null  object 
 7   deal_campaign        16399 non-null  object 
 8   sla                  15518 non-null  object 
 9   content              14125 non-null  object 
 10  term                 12431 non-null  object 
 11  source               21564 non-null  object 
 12  payment_type         495 non-null    object 
 13  product              3592 non-null   object 
 14  education_type       3299 non-null   object 
 15  created_time         21564 non-null  obje

### offer_total_amount

In [115]:
deals['offer_total_amount'].unique()

array([nan, 2000, 9000, 11000, 3500, 4500, '€ 2.900,00', 6500, 4000, 3000,
       10000, 2500, 5000, 11500, 1, 1000, 1200, 0, 1500, '€ 11398,00',
       11111, 6000], dtype=object)

In [116]:
deals['offer_total_amount'] = (
    deals['offer_total_amount']
    .astype(str)
    .str.replace('€', '')
    .str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.',)  
    .replace('nan', np.nan))

In [117]:
# Converting offer_total_amount to numeric type, invalid values will be replaced with NaN, then converting to Int64 to preserve missing values

deals['offer_total_amount'] = pd.to_numeric(deals['offer_total_amount'], errors='coerce')
deals['offer_total_amount'] = deals['offer_total_amount'].astype('Int64')

In [118]:
deals['offer_total_amount'].unique()

<IntegerArray>
[ <NA>,  2000,  9000, 11000,  3500,  4500,  2900,  6500,  4000,  3000, 10000,
  2500,  5000, 11500,     1,  1000,  1200,     0,  1500, 11398, 11111,  6000]
Length: 22, dtype: Int64

### payment_type

In [119]:
deals['payment_type'].unique()

array([nan, 'One Payment', 'Recurring Payments', 'Reservation'],
      dtype=object)

In [120]:
payment_type_nan = deals[deals['payment_type'].isnull()]
nan_payment_stages = payment_type_nan['stage'].value_counts(dropna=False)
print(nan_payment_stages)

stage
Lost                         15625
Call Delayed                  2237
Registered on Webinar         2071
Payment Done                   494
Waiting For Payment            322
Qualificated                   101
Registered on Offline Day      100
Need to Call - Sales            33
Need To Call                    31
Test Sent                       25
Need a consultation             23
New Lead                         6
Free Education                   1
Name: count, dtype: int64


In [121]:
# Filling missing values in payment_type with 'Unknown' for paid deals
deals.loc[(deals['stage'] == 'Payment Done') & (deals['payment_type'].isnull()), 'payment_type'] = 'Unknown'

In [122]:
# 479 values were filled as 'Unknown' in payment_type
# 20575 missing values remain

deals['payment_type'].value_counts(dropna=False)

payment_type
NaN                   20575
Unknown                 494
Recurring Payments      350
One Payment             140
Reservation               5
Name: count, dtype: int64

### education_type

In [123]:
deals['education_type'].value_counts(dropna=False)

education_type
NaN        18265
Morning     2895
Evening      404
Name: count, dtype: int64

In [124]:
# Filling missing values in education_type based on mode by product and offer_total_amount

deals['education_type'] = deals['education_type'].fillna(
    deals.groupby(['product', 'offer_total_amount'])['education_type']
         .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None))

In [125]:
# 140 missing values were filled

deals['education_type'].value_counts(dropna=False)

education_type
NaN        18125
Morning     3027
Evening      412
Name: count, dtype: int64

### product

In [126]:
deals['product'].unique()

array([nan, 'Web Developer', 'Digital Marketing', 'UX/UI Design',
       'Find yourself in IT', 'Data Analytics'], dtype=object)

In [127]:
deals['product'].value_counts(dropna=False)

product
NaN                    17972
Digital Marketing       1990
UX/UI Design            1022
Web Developer            575
Find yourself in IT        4
Data Analytics             1
Name: count, dtype: int64

In [128]:
# Filling missing values in product based on mode by education_type and offer_total_amount

deals['product'] = deals['product'].fillna(
    deals.groupby(['education_type', 'offer_total_amount'])['product']
         .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None))

In [129]:
# 4 missing values were filled

deals['product'].value_counts(dropna=False)

product
NaN                    17968
Digital Marketing       1994
UX/UI Design            1022
Web Developer            575
Find yourself in IT        4
Data Analytics             1
Name: count, dtype: int64

### course_duration

In [130]:
deals['course_duration'].value_counts(dropna=False)

course_duration
NaN     17977
11.0     3012
6.0       575
Name: count, dtype: int64

In [131]:
deals.groupby(['education_type', 'product'])['course_duration'].value_counts(dropna=False)

education_type  product            course_duration
Evening         Digital Marketing  11.0                257
                UX/UI Design       11.0                154
                Web Developer      6.0                   1
Morning         Digital Marketing  11.0               1648
                                   NaN                   4
                UX/UI Design       11.0                821
                Web Developer      6.0                 549
Name: count, dtype: int64

In [132]:
# Filling missing values in course_duration based on mode by education_type and product

deals['course_duration'] = deals['course_duration'].fillna(
    deals.groupby(['education_type', 'product'])['course_duration']
         .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else None))

In [133]:
# 4 missing values were replaced in course_duration

deals['course_duration'].value_counts(dropna=False)

course_duration
NaN     17973
11.0     3016
6.0       575
Name: count, dtype: int64

In [134]:
# One record out of 550 for Web Developer with education_type = Evening was found, it can be assumed that this is an error
# Replace Evening with Morning

deals.loc[(deals['education_type'] == 'Evening') & (deals['product'] == 'Web Developer')]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
10768,5805028000030164109,Victor Barnes,NaN,C - Low,Lost,Changed Decision,/eng,02.07.23wide_DE,06:47:21,b1de,...,Web Developer,Evening,05.02.2024 12:37,6.0,NaN,2000,2000,5805028000030143280,Nürnberg,NaN


In [135]:

deals.loc[(deals['education_type'] == 'Evening') & (deals['product'] == 'Web Developer'), 'education_type'] = 'Morning'

In [136]:
# checking
deals.loc[(deals['education_type'] == 'Evening') & (deals['product'] == 'Web Developer')]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch


### months_of_study

In [137]:
round(100 * deals.isnull().sum() / len(deals), 2)

deals_id                0.00
deal_manager            0.00
closing_date           32.12
quality                 0.00
stage                   0.00
lost_reason            25.04
page                    0.00
deal_campaign          23.95
sla                    28.04
content                34.50
term                   42.35
source                  0.00
payment_type           95.41
product                83.32
education_type         84.05
created_time            0.00
course_duration        83.35
months_of_study        96.10
initial_amount_paid    80.76
offer_total_amount     80.67
contact_name            0.28
city                   88.36
level_of_deutsch       94.20
dtype: float64

In [138]:
# This field will not be needed for further analysis since it is impossible to determine whether the student is still studying or not

deals = deals.drop('months_of_study', axis=1)

### contact_name

In [139]:
deals['contact_name'].isna().sum()

np.int64(60)

In [140]:
deals[deals['contact_name'].isna()]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
11246,5805028000029290052,Kevin Parker,17.02.2024,E - Non Qualified,Lost,Invalid number,/,NaN,00:59:43,NaN,...,NaN,NaN,NaN,31.01.2024 10:27,NaN,<NA>,<NA>,NaN,NaN,NaN
11247,5805028000029290031,Kevin Parker,17.02.2024,D - Non Target,Lost,Doesn't Answer,/,NaN,01:02:42,NaN,...,NaN,NaN,NaN,31.01.2024 10:24,NaN,<NA>,<NA>,NaN,NaN,NaN
11310,5805028000029010001,Quincy Vincent,31.01.2024,D - Non Target,Lost,Inadequate,/,NaN,00:09:29,NaN,...,NaN,NaN,NaN,30.01.2024 14:43,NaN,<NA>,<NA>,NaN,NaN,NaN
11363,5805028000028760241,Charlie Davis,15.02.2024,E - Non Qualified,Lost,Changed Decision,/direct,NaN,00:02:03,NaN,...,NaN,NaN,NaN,29.01.2024 17:50,NaN,<NA>,<NA>,NaN,NaN,NaN
11393,5805028000028690001,Victor Barnes,NaN,E - Non Qualified,Lost,Duplicate,/,NaN,00:16:37,NaN,...,NaN,NaN,NaN,29.01.2024 11:17,NaN,<NA>,<NA>,NaN,NaN,NaN
11789,5805028000028071113,Oliver Taylor,NaN,D - Non Target,Call Delayed,needs time to think,/direct,NaN,00:20:56,NaN,...,NaN,NaN,NaN,24.01.2024 14:58,NaN,<NA>,<NA>,NaN,NaN,NaN
11794,5805028000028111170,Victor Barnes,NaN,C - Low,Lost,Changed Decision,/,NaN,00:11:45,NaN,...,NaN,Web Developer,Morning,24.01.2024 14:22,6.0,1000,5000,NaN,Neumarkt in der Oberpfalz,B1
11796,5805028000028071053,Charlie Davis,NaN,C - Low,Payment Done,NaN,/direct,NaN,00:16:04,NaN,...,Unknown,Digital Marketing,Morning,24.01.2024 14:18,11.0,1000,11500,NaN,Kamenz,B1
11807,5805028000028109058,Victor Barnes,NaN,C - Low,Lost,Doesn't Answer,/,NaN,00:30:14,NaN,...,NaN,NaN,NaN,24.01.2024 12:24,NaN,<NA>,<NA>,NaN,NaN,NaN
11848,5805028000027944013,Oliver Taylor,23.01.2024,B - Medium,Lost,Went to Rivals,/direct,Akademia,NaN,NaN,...,NaN,NaN,NaN,23.01.2024 15:14,NaN,<NA>,<NA>,NaN,NaN,NaN


In [141]:
# We cannot delete these rows because they will be taken into account when calculating click conversion and profit
# Fill them with the value Unknown

deals['contact_name'] = deals['contact_name'].fillna('Unknown')

In [142]:
deals['contact_name'].isna().sum()

np.int64(0)

### city

In [143]:
deals['city'].value_counts()

city
-                348
Berlin           182
München           74
Hamburg           62
Leipzig           45
                ... 
Chuguev            1
Bad Hönningen      1
Senftenberg        1
Günzburg           1
Malschwitz         1
Name: count, Length: 876, dtype: int64

In [144]:
# Removing leading/trailing spaces and replacing ['', ' ', '-'] with NaN

deals['city'] = deals['city'].str.strip().replace(['', ' ', '-'], np.nan)

In [145]:
deals['contact_name'].value_counts()

contact_name
Unknown                60
5805028000003014152    54
5805028000005448163    39
5805028000017522090    19
5805028000014478367    13
                       ..
5805028000001076115     1
5805028000001079118     1
5805028000056849495     1
5805028000056875061     1
5805028000001520014     1
Name: count, Length: 18068, dtype: int64

In [146]:
# We see that contacts are repeated.
# Checking that for contacts with a filled city, the value is specified only in one record,
# and for duplicate contact_name the city remains empty

deals[deals['city'] != 'Unknown'].sort_values('contact_name').head(60)

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
20374,5805028000005180061,John Doe,NaN,E - Non Qualified,Registered on Webinar,NaN,/workshop,NaN,"3 days, 18:22:35",NaN,...,Recurring Payments,UX/UI Design,Morning,18.08.2023 19:58,11.0,100,4000,5805028000000872003,NaN,NaN
21142,5805028000003077037,Bob Brown,NaN,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,04.07.23recentlymoved_DE,NaN,v7com,...,NaN,NaN,NaN,25.07.2023 10:41,NaN,0,0,5805028000000872003,NaN,NaN
20717,5805028000004111157,John Doe,NaN,C - Low,Lost,Duplicate,/,NaN,"13 days, 23:40:53",NaN,...,One Payment,Web Developer,Morning,08.08.2023 14:40,6.0,3500,3500,5805028000000872003,NaN,NaN
20377,5805028000005176025,John Doe,NaN,F - Unknown,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,NaN,...,NaN,NaN,NaN,18.08.2023 19:42,NaN,<NA>,<NA>,5805028000000872003,NaN,NaN
20376,5805028000005168037,John Doe,NaN,F - Unknown,Registered on Webinar,NaN,/workshop,web2408_DE,NaN,NaN,...,NaN,NaN,NaN,18.08.2023 19:45,NaN,<NA>,<NA>,5805028000000872003,NaN,NaN
20375,5805028000005174051,John Doe,NaN,F - Unknown,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,NaN,NaN,18.08.2023 19:54,NaN,<NA>,<NA>,5805028000000872003,NaN,NaN
20370,5805028000005155037,John Doe,NaN,F - Unknown,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,Digital Marketing,Evening,18.08.2023 20:48,11.0,<NA>,<NA>,5805028000000872003,NaN,NaN
20371,5805028000005178088,John Doe,NaN,F - Unknown,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,NaN,NaN,18.08.2023 20:46,NaN,<NA>,<NA>,5805028000000872003,NaN,NaN
20373,5805028000005176076,John Doe,NaN,F - Unknown,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,Digital Marketing,Evening,18.08.2023 20:22,11.0,<NA>,<NA>,5805028000000872003,NaN,NaN
20884,5805028000003573412,Bob Brown,07.08.2023,E - Non Qualified,Lost,Duplicate,/test,NaN,NaN,NaN,...,NaN,NaN,NaN,02.08.2023 15:35,NaN,0,0,5805028000000872003,NaN,NaN


In [147]:
# There are contacts for which the city is filled, the value is specified only in one record, and for duplicate contact_name - the city remains empty! 
# Therefore, fill within the group (contact_name) with the first known city, if none — 'Unknown'
# Propagating cities by contact_name, but only for contact_name != 'Unknown' because for Unknown there may be several cities and 
# we will fill this group later

mask = deals['contact_name'] != 'Unknown'
deals.loc[mask, 'city'] = deals.loc[mask, 'city'].fillna(
    deals.loc[mask].groupby('contact_name')['city'].transform(
        lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None))

In [148]:
# If nothing is in the group — set 'Unknown'
deals['city'] = deals['city'].fillna('Unknown')

In [149]:
# Checking on a specific record

deals[deals['contact_name'] == '5805028000001073001']

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
21578,5805028000001040014,Quincy Vincent,14.07.2023,C - Low,Lost,Conditions are not suitable,eng/digital-marketing,02.07.23wide_DE,"8 days, 18:22:16",b3,...,NaN,Digital Marketing,Morning,05.07.2023 18:06,11.0,1000,11000,5805028000001073001,Brake,NaN
21579,5805028000001032016,Kevin Parker,05.07.2023,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,02.07.23wide_DE,NaN,b3,...,NaN,NaN,NaN,05.07.2023 18:05,NaN,<NA>,<NA>,5805028000001073001,Brake,NaN


In [150]:
deals['city'].isnull().sum()

np.int64(0)

### level_of_deutsch

In [151]:
deals['level_of_deutsch'].isna().sum()

np.int64(20313)

In [152]:
deals['level_of_deutsch'].value_counts()

level_of_deutsch
B1                                                      219
б1                                                      118
в1                                                      100
b1                                                       93
Б1                                                       93
                                                       ... 
точно уровень не знаю, но говорить могу - учила сама      1
А2-В1 учит                                                1
В1 (учится на В2 уже)                                     1
В январе - В2 сдает                                       1
b1 должна получить результаты в феврале                   1
Name: count, Length: 215, dtype: int64

In [153]:
# There are contacts for which the level_of_deutsch is filled, the value is specified only in one record, and for duplicate contact_name the level_of_deutsch remains empty! 
# Therefore, fill within the group (contact_name) with the first known specified level, if none — 'Unknown'
# Propagating level_of_deutsch by contact_name, but only for contact_name != 'Unknown' because for Unknown there may be different German levels and 
# we will fill this group later

mask_deutsch = deals['contact_name'] != 'Unknown'
deals.loc[mask_deutsch, 'level_of_deutsch'] = deals.loc[mask_deutsch, 'level_of_deutsch'].fillna(
    deals.loc[mask_deutsch].groupby('contact_name')['level_of_deutsch'].transform(
        lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None))

In [154]:
deals['level_of_deutsch'].unique()

array([nan, 'б1', 'в1', 'A2', 'в2', 'b1', 'В1', 'B1', 'в1-в2', 'А2-В1',
       'А2 ( Б1 в июне)', '-', 'B2', 'C2', 'с1', 'Б1', 'Б2', 'а2',
       'Б1( может будет)', 'а1', 'С1', 'a2 (b1 экзамен 15 июня)', 'а0',
       'б2', 'А2', 'B1 будет в феврале 2025',
       'Detmold, Paulinenstraße 95, 32756',
       'Сам оценивает на B2, 13 лет живет в Германии', 'В1-В2',
       'Б1 ( ждет Б2)',
       'lэкзамен - 6 июля на В1. курсы вечером (но уверенно говорит на B1)',
       'Гражданка Германии уже год в Германии Учит немецкий и в сентябре b1 через гос-во проходит, а не через ДЖЦ, вечером учится 3 р в неделю с 18 до 21',
       'B1 в процессе обучения',
       'ЯЗ: нем В1 был экз 03.05 повтор и сейчас ждет результаты. Технический англ был. А1 сейчас. ОБР: 2 во информационные и комп сети - инженер системоте',
       'В1 в сентябре', 'Нет', 0, 'Ждем B1', 'А1 сертиф, но по факту А2',
       'a2', 'Пока А2, сдает 17 05 B1', 'окончание 13.06 курса на b1',
       'A1', 'b2', 'В2', 'Thorn-Prikker-St

In [155]:
# Bring all values to a unified format and uppercase, normalize Cyrillic letters and fix typos (e.g., 'F' → 'A'). 
# If only '0' is found, replace with 'A0'.  
# Explicitly ignore English entries to simplify processing — normalization is done with a margin of error, 
# but there are only about 10 cases with simultaneous German and English, which is insignificant for analysis.  
# Extract only standard German language levels (A0–C2).  
# Replace all incorrect or empty values with 'Unknown' for ease of analysis and further categorization.

deals['level_of_deutsch_clean'] = (
    deals['level_of_deutsch']
    .astype(str)
    .str.upper()
    .str.replace('А', 'A')
    .str.replace('В', 'B')
    .str.replace('Б', 'B')
    .str.replace('С', 'C')
    .str.replace('F', 'A')
    .replace('0', 'A0') 
    .mask(lambda x: x.str.contains('АНГЛ|ENGLISH'))  # ignore English
    .str.extract(r'(A0|A1|A2|B1|B2|C1|C2)', expand=False)
    .fillna('Unknown')
)

In [156]:
# checking the result
deals[deals['level_of_deutsch'].notnull()]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch,level_of_deutsch_clean
57,5805028000056568397,Paula Underwood,NaN,F - Unknown,Registered on Webinar,NaN,/webinar,NaN,NaN,NaN,...,NaN,NaN,20.06.2024 12:17,NaN,<NA>,<NA>,5805028000020664131,Prenzlau,б1,B1
60,5805028000056558351,Ulysses Adams,NaN,C - Low,Waiting For Payment,NaN,/eng,NaN,00:09:49,NaN,...,Web Developer,Morning,20.06.2024 11:16,6.0,1000,9000,5805028000056578244,Dortmund,в1,B1
71,5805028000056564131,Ben Hall,NaN,D - Non Target,Waiting For Payment,NaN,/eng,20.05.24interests_DE,12:51:39,bloggersvideo14com,...,UX/UI Design,Morning,19.06.2024 22:31,11.0,1000,11000,5805028000056575100,München,A2,A2
89,5805028000056378468,Ulysses Adams,NaN,F - Unknown,Registered on Webinar,NaN,/webinar,webinar1906,NaN,NaN,...,NaN,NaN,19.06.2024 14:53,NaN,<NA>,<NA>,5805028000027735302,Berlin,в2,B2
98,5805028000056374135,Ulysses Adams,NaN,C - Low,Waiting For Payment,NaN,/eng,03.07.23women,02:42:33,v16,...,UX/UI Design,Morning,19.06.2024 12:12,11.0,1000,11000,5805028000056362225,Offenbach am Main,в1,B1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21367,5805028000001987082,Julia Nelson,17.07.2023,C - Low,Payment Done,Conditions are not suitable,eng/digital-marketing,12.07.2023wide_DE,00:49:16,v3com,...,Digital Marketing,Morning,17.07.2023 18:02,11.0,1000,11000,5805028000001986077,Unknown,b1 должна получить результаты в феврале,B1
21381,5805028000001880571,Bob Brown,21.07.2023,E - Non Qualified,Lost,Doesn't Answer,eng/digital-marketing,performancemax_digitalmarkt_ru_DE,NaN,_{region_name}_,...,NaN,NaN,17.07.2023 13:28,NaN,0,0,5805028000001855474,Unknown,Б1,B1
21423,5805028000001857134,Bob Brown,18.07.2023,E - Non Qualified,Lost,Doesn't Answer,eng/digital-marketing,performancemax_digitalmarkt_ru_DE,NaN,_{region_name}_,...,NaN,NaN,15.07.2023 02:16,NaN,0,0,5805028000001882098,Nürnberg,В1,B1
21456,5805028000001812001,Victor Barnes,14.07.2023,E - Non Qualified,Waiting For Payment,Non target,eng/digital-marketing,07.07.23LAL_DE,"311 days, 10:34:24",b2,...,Digital Marketing,Morning,14.07.2023 07:08,11.0,1000,11500,5805028000001782077,Unknown,B1,B1


In [157]:
# Overwriting the original column
deals['level_of_deutsch'] = deals['level_of_deutsch_clean']

# Deleting the temporary column
deals.drop(columns=['level_of_deutsch_clean'], inplace=True)

In [158]:
deals['level_of_deutsch'].value_counts()

level_of_deutsch
Unknown    19947
B1          1099
B2           223
A2           214
C1            36
A1            29
A0            12
C2             4
Name: count, dtype: int64

In [159]:
deals.isna().sum()

deals_id                   0
deal_manager               0
closing_date            6926
quality                    0
stage                      0
lost_reason             5399
page                       0
deal_campaign           5165
sla                     6046
content                 7439
term                    9133
source                     0
payment_type           20575
product                17968
education_type         18125
created_time               0
course_duration        17973
initial_amount_paid    17415
offer_total_amount     17395
contact_name               0
city                       0
level_of_deutsch           0
dtype: int64

## Converting data types for columns such as dates and numeric values.

In [160]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21564 entries, 0 to 21592
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   deals_id             21564 non-null  object 
 1   deal_manager         21564 non-null  object 
 2   closing_date         14638 non-null  object 
 3   quality              21564 non-null  object 
 4   stage                21564 non-null  object 
 5   lost_reason          16165 non-null  object 
 6   page                 21564 non-null  object 
 7   deal_campaign        16399 non-null  object 
 8   sla                  15518 non-null  object 
 9   content              14125 non-null  object 
 10  term                 12431 non-null  object 
 11  source               21564 non-null  object 
 12  payment_type         989 non-null    object 
 13  product              3596 non-null   object 
 14  education_type       3439 non-null   object 
 15  created_time         21564 non-null  obje

In [161]:
# dates
deals['closing_date'] = pd.to_datetime(deals['closing_date'], errors='coerce', dayfirst=True)
deals['created_time'] = pd.to_datetime(deals['created_time'], errors='coerce', dayfirst=True)
deals['course_duration'] = deals['course_duration'].astype('Int32')

# timedelta
deals['sla'] = pd.to_timedelta(deals['sla'])

# categories
for col in ['deal_manager', 'quality', 'stage', 'lost_reason', 'page', 'source', 'payment_type', 'product', 'education_type', 'city', 'level_of_deutsch']:
    deals[col] = deals[col].astype('category')

In [162]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21564 entries, 0 to 21592
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype          
---  ------               --------------  -----          
 0   deals_id             21564 non-null  object         
 1   deal_manager         21564 non-null  category       
 2   closing_date         14638 non-null  datetime64[ns] 
 3   quality              21564 non-null  category       
 4   stage                21564 non-null  category       
 5   lost_reason          16165 non-null  category       
 6   page                 21564 non-null  category       
 7   deal_campaign        16399 non-null  object         
 8   sla                  15518 non-null  timedelta64[ns]
 9   content              14125 non-null  object         
 10  term                 12431 non-null  object         
 11  source               21564 non-null  category       
 12  payment_type         989 non-null    category       
 13  product              

In [163]:
deals

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
0,5805028000056864695,Ben Hall,NaT,F - Unknown,New Lead,NaN,/eng/test,03.07.23women,NaT,v16,...,NaN,NaN,NaN,2024-06-21 15:30:00,<NA>,<NA>,<NA>,5805028000056849495,Unknown,Unknown
1,5805028000056859489,Ulysses Adams,NaT,F - Unknown,New Lead,NaN,/at-eng,NaN,NaT,NaN,...,NaN,Web Developer,Morning,2024-06-21 15:23:00,6,0,2000,5805028000056834471,Unknown,Unknown
2,5805028000056832357,Ulysses Adams,2024-06-21,D - Non Target,Lost,Non target,/at-eng,engwien_AT,0 days 00:26:43,b1-at,...,NaN,NaN,NaN,2024-06-21 14:45:00,<NA>,<NA>,<NA>,5805028000056854421,Unknown,Unknown
3,5805028000056824246,Eva Kent,2024-06-21,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,0 days 01:00:04,bloggersvideo14com,...,NaN,NaN,NaN,2024-06-21 13:32:00,<NA>,<NA>,<NA>,5805028000056889351,Unknown,Unknown
4,5805028000056873292,Ben Hall,2024-06-21,D - Non Target,Lost,Non target,/eng,discovery_DE,0 days 00:53:12,website,...,NaN,NaN,NaN,2024-06-21 13:21:00,<NA>,<NA>,<NA>,5805028000056876176,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21588,5805028000000970006,Jane Smith,2023-07-04,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,03.07.23women,NaT,b3,...,NaN,NaN,NaN,2023-07-04 07:10:00,<NA>,<NA>,<NA>,5805028000000979006,Unknown,Unknown
21589,5805028000000948010,Jane Smith,2023-08-29,B - Medium,Lost,needs time to think,eng/digital-marketing,03.07.23women,NaT,b3,...,NaN,NaN,NaN,2023-07-04 07:10:00,<NA>,<NA>,<NA>,5805028000000979006,Unknown,Unknown
21590,5805028000000945016,Jane Smith,2023-08-29,A - High,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,56 days 19:01:59,b3,...,NaN,NaN,NaN,2023-07-03 20:39:00,<NA>,<NA>,<NA>,5805028000000968001,Unknown,Unknown
21591,5805028000000927004,Bob Brown,2023-07-09,D - Non Target,Lost,Does not speak English,eng/digital-marketing,03.07.23women,NaT,b3,...,NaN,NaN,NaN,2023-07-03 20:17:00,<NA>,<NA>,<NA>,5805028000000961001,Unknown,Unknown


## Data validation: searching for illogical values

In [164]:
deals[deals['course_duration'] < 0]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch


No values less than 0 found

In [165]:
# Adding 23:59:59 to closing_date so that it can be logically compared with created_time

deals['closing_date'] = deals['closing_date'] + pd.Timedelta('23:59:59')

In [166]:
# Checking if there are records where created_time is later than closing_date, which is illogical

deals[deals['created_time'] > deals['closing_date']]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
454,5805028000055502890,Quincy Vincent,2024-06-11 23:59:59,D - Non Target,Lost,Changed Decision,/eng,24.09.23retargeting_DE,0 days 11:22:54,v15,...,NaN,NaN,NaN,2024-06-16 00:06:00,<NA>,<NA>,<NA>,5805028000055488584,Unknown,Unknown
2083,5805028000051847114,Quincy Vincent,2024-05-22 23:59:59,E - Non Qualified,Lost,Changed Decision,/eng/test,22.05.2024wide_DE,0 days 12:58:43,bloggersvideo18com,...,NaN,NaN,NaN,2024-05-25 21:29:00,<NA>,<NA>,<NA>,5805028000051866233,Unknown,Unknown
2787,5805028000049539444,Julia Nelson,2024-05-07 23:59:59,E - Non Qualified,Lost,Changed Decision,/eng/test,NaN,0 days 00:06:32,NaN,...,NaN,NaN,NaN,2024-05-12 11:19:00,<NA>,<NA>,<NA>,5805028000049438717,Unknown,Unknown
3019,5805028000048886321,Oliver Taylor,2024-05-07 23:59:59,C - Low,Payment Done,NaN,/,NaN,NaT,NaN,...,Recurring Payments,UX/UI Design,Morning,2024-05-08 15:31:00,11,1000,11000,5805028000048886280,Berlin,B1
3022,5805028000048883316,Ulysses Adams,2024-04-17 23:59:59,A - High,Lost,Duplicate,/eng,03.07.23women,NaT,v16,...,Recurring Payments,UX/UI Design,Morning,2024-05-08 14:48:00,11,1000,11000,5805028000043444319,Neu-Ulm,Unknown
3031,5805028000048886160,Oliver Taylor,2024-05-07 23:59:59,C - Low,Payment Done,NaN,/,NaN,NaT,NaN,...,Unknown,Digital Marketing,Morning,2024-05-08 12:54:00,11,1000,11000,5805028000048886140,Berlin,B1
3691,5805028000047482214,Paula Underwood,2024-04-23 23:59:59,E - Non Qualified,Lost,Expensive,/eng,12.07.2023wide_DE,0 days 00:24:45,bloggersvideo11,...,NaN,NaN,NaN,2024-04-30 15:16:00,<NA>,<NA>,<NA>,5805028000047492138,Unknown,Unknown
4107,5805028000046234250,Rachel White,2024-04-17 23:59:59,E - Non Qualified,Lost,Duplicate,/,NaN,NaT,NaN,...,NaN,NaN,NaN,2024-04-24 17:30:00,<NA>,<NA>,<NA>,5805028000046193190,Unknown,Unknown
4169,5805028000045961466,Paula Underwood,2024-04-18 23:59:59,D - Non Target,Lost,Expensive,/eng/test,12.07.2023wide_DE,0 days 20:38:10,bloggersvideo16com,...,NaN,NaN,NaN,2024-04-23 21:44:00,<NA>,<NA>,<NA>,5805028000045998344,Unknown,Unknown
4434,5805028000045314301,Quincy Vincent,2024-04-18 23:59:59,E - Non Qualified,Lost,Doesn't Answer,/eng,02.07.23wide_DE,0 days 05:17:10,bloggersvideo14com,...,NaN,NaN,NaN,2024-04-21 08:57:00,<NA>,<NA>,<NA>,5805028000045305375,Unknown,Unknown


In [167]:
# Given that only 43 records were found, the following solution is adopted:
# fixing created_time > closing_date: set closing_date = created_time + sla
# (if sla is nan then replace with 0 so that the date value in closing_date = created_time) 

deals.loc[deals['created_time'] > deals['closing_date'], 'closing_date'] = (
    deals['created_time'] + deals['sla'].fillna(pd.Timedelta(0))
).dt.normalize() + pd.Timedelta('23:59:59')                            # return to common format

In [168]:
deals[deals['created_time'] > deals['closing_date']]

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch


In [169]:
# Checking manually 
deals[deals['deals_id'] == '5805028000045245616']

,deals_id,deal_manager,closing_date,quality,stage,lost_reason,page,deal_campaign,sla,content,...,payment_type,product,education_type,created_time,course_duration,initial_amount_paid,offer_total_amount,contact_name,city,level_of_deutsch
4520,5805028000045245616,Ulysses Adams,2024-05-17 23:59:59,B - Medium,Lost,Doesn't Answer,eng/digital-marketing,performancemax_digitalmarkt_ru_DE,27 days 13:56:25,_{region_name}_,...,NaN,Digital Marketing,Morning,2024-04-20 06:19:00,11,1000,11000,5805028000003597256,Eggolsheim,Unknown


In [170]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21564 entries, 0 to 21592
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype          
---  ------               --------------  -----          
 0   deals_id             21564 non-null  object         
 1   deal_manager         21564 non-null  category       
 2   closing_date         14638 non-null  datetime64[ns] 
 3   quality              21564 non-null  category       
 4   stage                21564 non-null  category       
 5   lost_reason          16165 non-null  category       
 6   page                 21564 non-null  category       
 7   deal_campaign        16399 non-null  object         
 8   sla                  15518 non-null  timedelta64[ns]
 9   content              14125 non-null  object         
 10  term                 12431 non-null  object         
 11  source               21564 non-null  category       
 12  payment_type         989 non-null    category       
 13  product              

In [171]:
deals.isna().sum()

deals_id                   0
deal_manager               0
closing_date            6926
quality                    0
stage                      0
lost_reason             5399
page                       0
deal_campaign           5165
sla                     6046
content                 7439
term                    9133
source                     0
payment_type           20575
product                17968
education_type         18125
created_time               0
course_duration        17973
initial_amount_paid    17415
offer_total_amount     17395
contact_name               0
city                       0
level_of_deutsch           0
dtype: int64

In [172]:
# city
# checking for entries with addresses (where there are numbers, streets, or non-English characters)
dirty = deals[
    deals['city'].str.contains(r'\d+|str\.|straße', case=False, na=False) |
    deals['city'].str.contains(r'[^\x00-\x7Fа-яА-ЯёЁ]', na=False)  # non-ASCII characters (German letters)
]

print(f"Found {len(dirty)} address entries")
dirty[['city']].head(20)

Found 516 address entries


,city
71,München
82,München
114,Görlitz
137,"Karl-Liebknecht str. 24, Hildburghausen, Thüri..."
148,Düsseldorf
155,Düsseldorf
171,Rüdesheim am Rhein
254,Düsseldorf
421,München
494,"Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a"


In [173]:
# Checking if there are records where the city name contains numbers (house numbers, postal codes, apartment numbers, etc.)

dirty_numbers = deals[deals['city'].str.contains(r'\d+', na=False)]
print(f"Records with numbers: {len(dirty_numbers)}")
dirty_numbers[['city']].head(10)

Records with numbers: 5


,city
137,"Karl-Liebknecht str. 24, Hildburghausen, Thüri..."
494,"Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a"
523,"Karl-Liebknecht str. 24, Hildburghausen, Thüri..."
872,"Vor Ebersbach 1, 77761 Schiltach"
1211,"Vor Ebersbach 1, 77761 Schiltach"


#### In this case, 5 "dirty" records were found, but fixing them manually is not practical.
#### It is better to implement a universal function that will automatically extract the city name from the text.
#### Records related to other countries will remain unchanged.

In [174]:
# Loading a pre-prepared list of German cities
with open('DE.txt', 'r', encoding='utf-8') as f:
    de_cities_list = [c.strip() for c in f.read().split(',')]

In [175]:
# Sorting from longest to shortest to avoid false matches of short words inside longer names (e.g., 'Regen' inside 'Regensburg')
de_cities_list = sorted(de_cities_list, key=len, reverse=True)

In [176]:
# Function to find city in the list

def find_city_final(text):
    if text == 'Unknown' or not isinstance(text, str):
        return text
    
    for city in de_cities_list:
        # We look for the city so that there are NO letters or digits around it. 
        # This will prevent "Stade" from matching inside "Stadecken", 
        # but will allow "Stadecken-Elsheim" to be matched completely.
        pattern = rf'(?<![a-zA-Z0-9]){re.escape(city)}(?![a-zA-Z0-9])'
        
        if re.search(pattern, text, re.IGNORECASE):
            return city
    return text

In [177]:
# The process took 8 minutes
deals['city_clean'] = deals['city'].apply(find_city_final)

In [178]:
# Checking how many have changed
print(f'Total {(deals["city"] != deals["city_clean"]).sum()} corrections in names ')
print(deals[deals["city"] != deals["city_clean"]][["city", "city_clean"]])

Total 15 corrections in names 
                                                    city      city_clean
137    Karl-Liebknecht str. 24, Hildburghausen, Thüri...  Hildburghausen
523    Karl-Liebknecht str. 24, Hildburghausen, Thüri...  Hildburghausen
872                     Vor Ebersbach 1, 77761 Schiltach       Schiltach
1211                    Vor Ebersbach 1, 77761 Schiltach       Schiltach
1606                          Bad Wildbad im Schwarzwald     Bad Wildbad
5448                                      Mainz-Kostheim           Mainz
5493                                        Mainz-Kastel           Mainz
7920                                        Mainz-Kastel           Mainz
7939                             Alzenau in Unterfranken         Alzenau
11243                                     Mainz-Kostheim           Mainz
13738                            Alzenau in Unterfranken         Alzenau
14558                              Kirchheim bei München         München
16447               

#### District names (such as Mainz-Kastel) have also been replaced with the main city name (Mainz), etc.

In [179]:
# Replacing the old column with cleaned data
deals['city'] = deals['city_clean']

In [180]:
# Deleting the temporary column city_clean to avoid duplicating data in memory
deals = deals.drop(columns=['city_clean'])

In [181]:
# Converting back to categorical type
deals['city'] = deals['city'].astype('category')

In [182]:
# Final type check before export
deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21564 entries, 0 to 21592
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype          
---  ------               --------------  -----          
 0   deals_id             21564 non-null  object         
 1   deal_manager         21564 non-null  category       
 2   closing_date         14638 non-null  datetime64[ns] 
 3   quality              21564 non-null  category       
 4   stage                21564 non-null  category       
 5   lost_reason          16165 non-null  category       
 6   page                 21564 non-null  category       
 7   deal_campaign        16399 non-null  object         
 8   sla                  15518 non-null  timedelta64[ns]
 9   content              14125 non-null  object         
 10  term                 12431 non-null  object         
 11  source               21564 non-null  category       
 12  payment_type         989 non-null    category       
 13  product              

# Export preserving all data types

In [183]:
crm_clean_data = [calls, contacts, spend, deals]

with open('crm_clean_data.pickle', 'wb') as f:
    pickle.dump(crm_clean_data, f)